# Recuperador KNN / Denso (TF-IDF) --- etapa (d)

**Disciplina:** Inteligência Artificial --- FACOM/UFMS --- 2026/1

Este notebook implementa o segundo recuperador obrigatório: a ideia de KNN aplicada à
**recuperação** (não classificação). Cada documento e cada consulta são representados
como vetores TF-IDF; o ranking é dado pelos *K* documentos com maior similaridade do
cosseno em relação à consulta.

Reaproveita o mesmo pré-processamento de `src/preprocessing.py` usado no BM25
(`02_baseline_bm25.ipynb`), para que ambos os recuperadores operem sobre representações
comparáveis do texto.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sys.path.append("..")
from src.preprocessing import preprocess_to_string

## 1. Carregamento do corpus

In [2]:
CORPUS_PATH = Path("../data/corpus.jsonl")

docs = []
with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        docs.append(json.loads(line))

print(f"Documentos carregados: {len(docs)}")

Documentos carregados: 1097


## 2. Vetorização TF-IDF

Cada documento (título + abstract, pré-processado) se torna um vetor onde cada dimensão é
um termo do vocabulário; o valor é alto quando o termo é frequente no documento (TF) e raro
no corpus (IDF). A consulta é vetorizada com o **mesmo vocabulário** aprendido na coleção
(`vectorizer.transform`, não `fit_transform`), para que ambos os lados vivam no mesmo espaço.

In [3]:
texts_preprocessed = [
    preprocess_to_string(d["title"] + ". " + d["abstract"]) for d in docs
]

vectorizer = TfidfVectorizer(min_df=2, max_df=0.85)
doc_vectors = vectorizer.fit_transform(texts_preprocessed)

print("Matriz TF-IDF:", doc_vectors.shape, "(documentos x termos do vocabulário)")

Matriz TF-IDF: (1097, 3870) (documentos x termos do vocabulário)


## 3. Função de busca (KNN via similaridade do cosseno)

Para cada consulta: (1) aplica o mesmo pré-processamento, (2) vetoriza com o vocabulário
da coleção, (3) calcula a similaridade do cosseno contra todos os documentos, (4) devolve
os *K* documentos com maior similaridade --- exatamente como uma busca por vizinhos mais
próximos, mas devolvendo o ranking em vez de votar uma classe.

In [4]:
def search(query: str, k: int = 10):
    q_text = preprocess_to_string(query)
    q_vector = vectorizer.transform([q_text])
    sims = cosine_similarity(q_vector, doc_vectors).flatten()
    top_idx = sims.argsort()[::-1][:k]
    return [(int(i), float(sims[i]), docs[i]) for i in top_idx]

In [5]:
query = "subgraph matching algorithms for graph databases"
results = search(query, k=10)

for rank, (idx, score, d) in enumerate(results, 1):
    print(f"{rank:>2}. [{score:.4f}] {d['title']}")
    print(f"     {d['arxiv_id']} | {d.get('primary_category')} | {d.get('published','')[:10]}")
    print(f"     {d['abstract'][:200]}...")
    print()

 1. [0.5459] GNN-based Anchor Embedding for Efficient Exact Subgraph Matching
     2502.00031 | cs.SI | 2025-01-23
     Subgraph matching query is a fundamental problem in graph data management and has a variety of real-world applications. Several recent works utilize deep learning (DL) techniques to process subgraph m...

 2. [0.5365] Neural Subgraph Matching
     2007.03092 | cs.LG | 2020-07-06
     Subgraph matching is the problem of determining the presence and location(s) of a given query graph in a large target graph. Despite being an NP-complete problem, the subgraph matching problem is cruc...

 3. [0.4717] A customizable inexact subgraph matching algorithm for attributed graphs
     2512.04280 | cs.DS | 2025-12-03
     Graphs provide a natural way to represent data by encoding information about objects and the relationships between them. With the ever-increasing amount of data collected and generated, locating speci...

 4. [0.4589] Uncertainty-aware Efficient Subgraph Isomorp

## 4. Gerando o arquivo `runs/knn.trec` para avaliação

Mesmo formato TREC usado pelo BM25: `qid Q0 doc_id rank score system`.

In [6]:
queries_path = Path("../eval/queries.tsv")
runs_dir = Path("./runs"); runs_dir.mkdir(exist_ok=True)
run_path = runs_dir / "knn.trec"

queries = pd.read_csv(queries_path, sep="\t", names=["qid", "text"])
with open(run_path, "w", encoding="utf-8") as f:
    for _, q in queries.iterrows():
        for rank, (idx, score, d) in enumerate(search(q["text"], k=100), 1):
            f.write(f"{q['qid']} Q0 {d['arxiv_id']} {rank} {score:.6f} knn\n")

print("Run salva em:", run_path)

Run salva em: runs/knn.trec


In [7]:
!head -n 5 {run_path}

q1 Q0 2502.00031 1 0.545912 knn
q1 Q0 2007.03092 2 0.536524 knn
q1 Q0 2512.04280 3 0.471733 knn
q1 Q0 2209.09090 4 0.458901 knn
q1 Q0 2104.00186 5 0.450084 knn


## 5. Próximos passos

1. Construir `eval/qrels.tsv` via *pooling* do top-k de `bm25.trec` e `knn.trec`.
2. Rodar `eval/evaluate.py` comparando os dois *runs*.
3. Implementar o módulo M5 (ranking híbrido sparse+dense) combinando os scores deste
   notebook com os do BM25.